In [1]:
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat ({df_cabang.shape[0]} baris)")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat (200 baris)
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat (200 baris)
Berkas 'transaksi_semarang.csv' berhasil dibuat (200 baris)

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [3]:
# Membuat direktori raw dan processed di HDFS
!hdfs dfs -mkdir -p /user/$USER/ecommerce/raw
!hdfs dfs -mkdir -p /user/$USER/ecommerce/processed

# Bukti direktori telah terbentuk
!hdfs dfs -ls /user/$USER/ecommerce

Found 2 items
drwxr-xr-x   - kaky supergroup          0 2026-09-10 07:53 /user/kaky/ecommerce/processed
drwxr-xr-x   - kaky supergroup          0 2026-09-10 08:21 /user/kaky/ecommerce/raw


In [4]:
# Upload 3 berkas CSV ke direktori HDFS raw
!hdfs dfs -put -f transaksi_magelang.csv /user/$USER/ecommerce/raw/
!hdfs dfs -put -f transaksi_yogyakarta.csv /user/$USER/ecommerce/raw/
!hdfs dfs -put -f transaksi_semarang.csv /user/$USER/ecommerce/raw/

# Tampilkan isi direktori beserta ukuran berkasnya
!hdfs dfs -ls -h /user/$USER/ecommerce/raw

Found 3 items
-rw-r--r--   1 kaky supergroup     12.0 K 2026-09-10 08:32 /user/kaky/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 kaky supergroup     11.9 K 2026-09-10 08:32 /user/kaky/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 kaky supergroup     12.4 K 2026-09-10 08:32 /user/kaky/ecommerce/raw/transaksi_yogyakarta.csv


In [7]:
import io
import os
import subprocess
import pandas as pd

def baca_csv_langsung_dari_hdfs(hdfs_path):
    """Membaca isi berkas di HDFS secara streaming menggunakan stdout tanpa menyentuh disk lokal."""
    # Otomatis menerjemahkan $USER menjadi username sistem tanpa perlu bikin variabel
    path_asli = os.path.expandvars(hdfs_path)

    cmd = ["hdfs", "dfs", "-cat", path_asli]
    proses = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = proses.communicate()
    if proses.returncode != 0:
        raise RuntimeError(f"Gagal membaca file dari HDFS: {stderr}")
    return pd.read_csv(io.StringIO(stdout))

# Langsung pakai $USER di dalam teks (tanpa variabel perantara!)
df_magelang = baca_csv_langsung_dari_hdfs("/user/$USER/ecommerce/raw/transaksi_magelang.csv")
df_jogja    = baca_csv_langsung_dari_hdfs("/user/$USER/ecommerce/raw/transaksi_yogyakarta.csv")
df_semarang = baca_csv_langsung_dari_hdfs("/user/$USER/ecommerce/raw/transaksi_semarang.csv")

# Menggabungkan ketiga DataFrame menjadi satu
df_gabungan = pd.concat([df_magelang, df_jogja, df_semarang], ignore_index=True)

print(f"Total baris gabungan: {len(df_gabungan)}")
print("\nDistribusi Transaksi per Cabang (value_counts):")
print(df_gabungan["kota"].value_counts())

# Tampilkan 5 baris teratas
df_gabungan.head()

Total baris gabungan: 600

Distribusi Transaksi per Cabang (value_counts):
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


In [8]:
# 1. Kalkulasi kolom total_pendapatan
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]

# 2. Agregasi pendapatan per kota dan kategori
df_ringkasan = (
    df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
    .sort_values(by=["kota", "total_pendapatan"], ascending=[True, False])
)

print("=== RINGKASAN PENDAPATAN PER KOTA & KATEGORI ===")
display(df_ringkasan.head(10))

# 3. Simpan berkas hasil olahan ke disk lokal sementara
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

# 4. Unggah kedua berkas hasil ke folder 'processed' di HDFS
!hdfs dfs -put -f data_gabungan_bersih.csv /user/$USER/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/$USER/ecommerce/processed/

# Verifikasi isi direktori processed
!hdfs dfs -ls -h /user/$USER/ecommerce/processed

=== RINGKASAN PENDAPATAN PER KOTA & KATEGORI ===


,kota,kategori,total_pendapatan
1,Magelang,Fashion,27750000
0,Magelang,Elektronik,18775000
3,Magelang,Makanan & Minuman,17525000
2,Magelang,Kesehatan & Kecantikan,17375000
4,Magelang,Rumah Tangga,12200000
6,Semarang,Fashion,26425000
5,Semarang,Elektronik,21425000
8,Semarang,Makanan & Minuman,19750000
7,Semarang,Kesehatan & Kecantikan,13525000
9,Semarang,Rumah Tangga,10975000


Found 2 items
-rw-r--r--   1 kaky supergroup     40.3 K 2026-09-10 08:39 /user/kaky/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 kaky supergroup        530 2026-09-10 08:39 /user/kaky/ecommerce/processed/ringkasan_kota_kategori.csv
